In [0]:
dbutils.widgets.removeAll()

In [0]:

from pyspark.sql.functions import *
from pyspark.sql.types import *


In [0]:

dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_au")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsssmartdata2698")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/product_category_name_translation.csv"

In [0]:
df_category_translation = spark.read.option("header", True)\
                                    .option("inferSchema", True)\
                                    .csv(ruta)

In [0]:
category_translation_schema = StructType(fields=[
    StructField("product_category_name", StringType(), True),
    StructField("product_category_name_english", StringType(), True)
])

In [0]:
df_category_translation_final = spark.read\
.option("header", True)\
.schema(category_translation_schema)\
.csv(ruta)

In [0]:
category_translation_selected_df = df_category_translation_final.select(
    col("product_category_name"),
    col("product_category_name_english")
)

In [0]:
category_translation_renamed_df = category_translation_selected_df

In [0]:
category_translation_final_df = category_translation_renamed_df.withColumn("ingestion_date", current_timestamp())

In [0]:
category_translation_final_df.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.category_translation_raw")